# Product recommendation system

**Goal:** recommend electronics each shopper is likely to rate highly, based on their past ratings and on what similar shoppers liked.

I compare four approaches on the same ratings data: popularity, user-user and item-item collaborative filtering (KNN), and matrix factorization (SVD). The reusable code lives in `src/recommender/`; this notebook walks through the comparison and the reasoning.

**Data:** the public Amazon Electronics ratings file (about 7.8 million ratings). See `data/README.md`; it is not committed. Set `RECOMMENDER_SYNTHETIC=1` to run on generated data.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from surprise import SVD, KNNBasic

from recommender import config
from recommender.data import filter_active, load_ratings
from recommender.evaluate import evaluate
from recommender.models import knn, knn_from_params, split, svd, to_dataset, tune
from recommender.popularity import product_stats, top_n_popular
from recommender.recommend import layered_recommend, recommend
from recommender.synthetic import make_ratings

sns.set_theme(style="whitegrid")
print("Split seed:", config.SPLIT_SEED, "| Model seed:", config.MODEL_SEED)

## Load and filter

In [ ]:
DATA_PATH = Path(os.environ.get("RECOMMENDER_DATA", "../data/raw/ratings_Electronics.csv"))
SYNTHETIC = os.environ.get("RECOMMENDER_SYNTHETIC") == "1"

raw = make_ratings() if SYNTHETIC else load_ratings(DATA_PATH)
print(f"{len(raw):,} raw ratings")

Most shoppers rate only a handful of products and most products have only a few ratings. Collaborative filtering needs overlap between shoppers, and the full matrix is too big to model on a laptop. So I keep **shoppers with at least 50 ratings** and, among those, **products with at least 5 ratings**.

In [ ]:
ratings = filter_active(raw)
print(
    f"{len(ratings):,} ratings from {ratings['user_id'].nunique():,} users "
    f"on {ratings['prod_id'].nunique():,} products"
)
print("Missing values:", int(ratings.isna().sum().sum()))
ratings["rating"].describe().to_frame().T

Original run: **65,290 ratings from 1,540 users on 5,689 products**, no missing values. Ratings skew high: mean 4.29, median 5. That skew matters later: a model that always predicts the average would already clear the 3.5 relevance threshold.

In [ ]:
ax = (ratings["rating"].value_counts(normalize=True).sort_index() * 100).plot(
    kind="bar", color="#3274A1"
)
ax.set(title="Distribution of ratings", xlabel="Rating", ylabel="% of ratings")
plt.show()

per_user = ratings.groupby("user_id").size()
print("Ratings per user: median", per_user.median(), "| max", per_user.max())

## Model 1: popularity

Rank products by average rating, but only among products with a minimum number of ratings, so one five-star review can't top the list. This is the only model that works for a brand-new shopper.

In [ ]:
stats = product_stats(ratings)
display_cols = ["avg_rating", "rating_count"]
print("At least 50 ratings:")
print(top_n_popular(stats, n=5, min_ratings=50)[display_cols].round(3))
print("At least 100 ratings:")
print(top_n_popular(stats, n=5, min_ratings=100)[display_cols].round(3))

## How I evaluate the personalized models

- **Split:** 80% of ratings for training, 20% for testing (fixed seed).
- **RMSE:** how far predicted ratings are from true ratings.
- **Precision@10, recall@10, F1@10:** for each shopper, take the 10 test products with the highest predicted rating. A product is *recommended* if its prediction is at least 3.5 and *relevant* if its true rating is at least 3.5. Average over shoppers.
- **Fallback share:** the share of predictions where a KNN model had too few neighbors and returned the global average instead. I added this after finding the problem described below; it wasn't measured in the original run.

In [ ]:
data = to_dataset(ratings)
trainset, testset = split(data)
results = {}


def score(name, model):
    model.fit(trainset)
    results[name] = evaluate(model, testset).as_dict()
    return results[name]

## Model 2: user-user collaborative filtering (KNN)

In [ ]:
user_knn = knn(similarity="cosine", user_based=True)
score("User-user KNN, baseline (cosine)", user_knn)

In [ ]:
user_params, user_cv_rmse = tune(KNNBasic, config.USER_KNN_GRID, data)
print(user_params, round(user_cv_rmse, 4))
user_knn_tuned = knn_from_params(user_params)
score("User-user KNN, tuned", user_knn_tuned)

Original run: tuning chose k = 50, min_k = 6, cosine similarity. RMSE improved from 1.0012 to 0.9526 and F1@10 from 0.856 to 0.870.

## Model 3: item-item collaborative filtering (KNN)

In [ ]:
item_knn = knn(similarity="cosine", user_based=False)
score("Item-item KNN, baseline (cosine)", item_knn)

In [ ]:
item_params, item_cv_rmse = tune(KNNBasic, config.ITEM_KNN_GRID, data)
print(item_params, round(item_cv_rmse, 4))
item_knn_tuned = knn_from_params(item_params)
score("Item-item KNN, tuned", item_knn_tuned)

Original run: tuning chose k = 30, min_k = 6, mean squared difference (MSD) similarity. RMSE improved from 0.9950 to 0.9576 and F1@10 from 0.841 to 0.859.

## Model 4: matrix factorization (SVD)

In [ ]:
score("SVD, baseline", svd())

In [ ]:
svd_params, svd_cv_rmse = tune(SVD, config.SVD_GRID, data)
print(svd_params, round(svd_cv_rmse, 4))
svd_tuned = svd(**svd_params)
score("SVD, tuned", svd_tuned)

Original run: tuning chose 20 epochs, learning rate 0.01, regularization 0.2. RMSE 0.8808, the best of all models; F1@10 0.866, essentially the same as the untuned SVD.

## Compare

In [ ]:
pd.DataFrame(results).T

## The finding that changed the design

Averages hide a problem. When a KNN model can't find `min_k` neighbors it doesn't fail: it returns the **global average rating** and flags the prediction as impossible. Because the average (4.29 in the original run) is above the 3.5 threshold, these generic guesses still count as "relevant" in precision and recall.

In the original run, the tuned user-user model fell back to the average for both example user-product pairs I checked, and the tuned item-item model's top five products for one example shopper were all the same fallback value. Below I look at the share of fallbacks across the whole test set, and at one shopper's top five.

In [ ]:
pd.DataFrame(results).T[["fallback_share"]]

In [ ]:
example_user = ratings["user_id"].value_counts().index[len(per_user) // 2]  # a typical shopper
for name, model in [("Item-item KNN, tuned", item_knn_tuned), ("SVD, tuned", svd_tuned)]:
    recs = recommend(model, ratings, example_user, n=5)
    flags = [
        model.predict(example_user, r.prod_id).details.get("was_impossible", False) for r in recs
    ]
    print(name)
    print(pd.DataFrame({"score": [round(r.score, 3) for r in recs], "fallback": flags}))

## Recommendation: a layered recommender

- **Tuned SVD as the main personalized model:** the best rating accuracy, and it always produces a real personal estimate.
- **Item-item KNN for explainable "customers who liked this also liked..." picks,** but only where it has real neighbors. Fallback predictions are dropped and SVD fills the slot.
- **Popularity for new or anonymous shoppers,** with a minimum-ratings rule.

The tuned user-user model had the highest F1@10 (0.870 vs 0.866 for SVD), but the lead is small, and it falls back to the generic average for exactly the shoppers and products where a recommendation matters most.

`layered_recommend` implements this design. It is not evaluated here; testing it properly needs an online experiment.

In [ ]:
popular = list(top_n_popular(stats, n=10, min_ratings=50).index)
layered = layered_recommend(
    ratings, example_user, primary=svd_tuned, explainable=item_knn_tuned, popular=popular, n=10
)
pd.DataFrame([{"score": round(r.score, 3), "source": r.source} for r in layered])

## Known weaknesses

- The grid searches ran on the full filtered dataset, so their folds overlap the test split. The test scores of the tuned models are somewhat optimistic. Next time I'd tune on the training split only.
- A random split lets the model see a shopper's future ratings. A time-based split would match real use.

## What I'd do next

- Add a "predict the average" baseline. With ratings this high it would score well on precision and recall, and every model must beat it.
- Evaluate on a time-based split (train on older ratings, test on newer).
- Measure catalog coverage and novelty, not just accuracy, so the system doesn't keep showing the same bestsellers.
- Test the layered design online with an A/B test on click-through and purchases.